# QLoRA fine-tuning on Kaggle — invoice extraction

Trains `Qwen/Qwen2-VL-2B-Instruct` with 4-bit QLoRA on **CORD-v2 receipts + synthetic
French/Moroccan invoices**, and produces a LoRA adapter of roughly 74 MB.

**Why Kaggle and not the local GPU:** the laptop card is an RTX 4050 with 6 GB. Inference
in 4-bit fits there comfortably (~1.7 GB) and all evaluation is done locally, but a training
step also holds activations, gradients and optimiser state. Kaggle's free tier gives a
P100 (16 GB) or 2×T4, which is enough, for about 30 GPU-hours a week.

## Before you start

1. **Settings → Accelerator → GPU P100** (preferred) or **GPU T4 x2**.
2. **Settings → Internet → On** (needed to download the model and CORD-v2).
3. Expect **roughly 2.5 hours** for one epoch over 1000 examples. Kaggle sessions last 12 h,
   so this fits, but do not close the browser tab for the first few minutes — if it is going
   to crash, it crashes at the first training step.

## What to do at the end

The last cell zips the adapter to `/kaggle/working/qwen2vl-2b-lora-v2.zip`. Download it from
the **Output** tab on the right, and unzip it into `checkpoints/` in the repo. Then run the
evaluation locally:

```bash
python -m invoice_extraction.baseline --dataset cord \
    --adapter checkpoints/qwen2vl-2b-lora-v2 --out results/finetuned_cord.json
python -m invoice_extraction.baseline --dataset synthetic \
    --adapter checkpoints/qwen2vl-2b-lora-v2 --out results/finetuned_synthetic.json
```

## 1 · Environment check

Confirm which GPU was actually allocated before spending an hour on it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2 · Install dependencies

Kaggle images already carry torch and transformers. We add the QLoRA stack and the
rendering stack used by the synthetic invoice generator.

`weasyprint` needs a couple of system libraries that the Kaggle image does not always have,
hence the `apt-get` line.

In [ ]:
!apt-get -qq update && apt-get -qq install -y libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz-subset0 > /dev/null 2>&1
!pip install -q -U "transformers>=4.57.0" peft bitsandbytes accelerate datasets
!pip install -q faker weasyprint pymupdf albumentations jinja2 pyyaml
print("dependencies installed")

## 3 · Get the project code

Cloning keeps the notebook honest: the training code that runs here is exactly the code in
the repository, not a copy that has drifted.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Israe236/invoice-extraction.git"
REPO_DIR = "/kaggle/working/invoice-extraction"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("working directory:", os.getcwd())
!git log --oneline -1

## 4 · Training configuration

Read from `configs/train.yaml` so the notebook has no hyperparameters of its own. The
settings that matter, and why:

| Setting | Value | Reason |
|---|---|---|
| `max_pixels` | 401408 | ~512 visual tokens. At 1003520 a single step took ~217 s, which is ~18 h for one epoch — more than a Kaggle session. |
| `per_device_train_batch_size` | 1 | One image per step; visual tokens dominate memory. |
| `gradient_accumulation_steps` | 8 | Recovers an effective batch of 8 without the memory of one. |
| `optim` | `paged_adamw_8bit` | 8-bit optimiser state, and paging survives a transient spike instead of OOMing. |
| `lora_r` / `lora_alpha` | 16 / 32 | Standard 2:1 ratio; ~1 % of parameters trained. |
| `num_synthetic_examples` | 200 | Mixed with 800 CORD receipts. CORD has no ICE, IF, date or currency, so without these the model cannot learn those fields at all. |

In [ ]:
!cat configs/train.yaml

## 5 · Smoke test (~5 minutes)

**Do not skip this.** It runs the full pipeline for 3 optimiser steps. Every bug that has
cost a session so far — the collator receiving stripped columns, `DataParallel` corrupting
4-bit weights, a resolution that made a step take four minutes — surfaces here, five minutes
in, instead of two hours in.

In [ ]:
import yaml

with open("configs/train.yaml") as f:
    smoke = yaml.safe_load(f)

smoke["max_steps"] = 3
smoke["num_synthetic_examples"] = 4  # rendering 200 invoices takes minutes on its own
smoke["output_dir"] = "/kaggle/working/smoke_test"
smoke["logging_steps"] = 1

with open("/kaggle/working/smoke.yaml", "w") as f:
    yaml.safe_dump(smoke, f)

from invoice_extraction.train import train

train("/kaggle/working/smoke.yaml")
print("\nSmoke test passed — the pipeline runs end to end.")

## 6 · The real run (~2.5 hours)

Only run this once the smoke test above has passed.

The first few minutes render 200 synthetic invoices before any training step happens, so a
quiet period at the start is expected, not a hang.

In [ ]:
import time

started = time.time()
train("configs/train.yaml")
print(f"\nTotal training time: {(time.time() - started) / 3600:.2f} hours")

## 7 · Loss curve

A loss that drops and then stays flat is what we want. A loss that spikes or goes to `nan`
means the learning rate is too high for this setup.

In [ ]:
import json

import matplotlib.pyplot as plt

with open("checkpoints/qwen2vl-2b-lora-v2/log_history.json") as f:
    history = json.load(f)

steps = [entry["step"] for entry in history if "loss" in entry]
losses = [entry["loss"] for entry in history if "loss" in entry]

plt.figure(figsize=(9, 4))
plt.plot(steps, losses, linewidth=1.4)
plt.xlabel("optimiser step")
plt.ylabel("training loss")
plt.title("QLoRA fine-tuning — Qwen2-VL-2B on CORD-v2 + synthetic invoices")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/loss_curve.png", dpi=140)
plt.show()

print(f"first logged loss: {losses[0]:.4f}")
print(f"last  logged loss: {losses[-1]:.4f}")

## 8 · Package the adapter for download

Only the adapter is saved, not the base model. That is the whole point of LoRA: the 4-bit
base weights are unchanged and can be re-downloaded from Hugging Face, so the artefact that
has to move between machines is ~74 MB rather than ~4.4 GB.

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/qwen2vl-2b-lora-v2", "zip", "checkpoints/qwen2vl-2b-lora-v2"
)
shutil.copy("checkpoints/qwen2vl-2b-lora-v2/log_history.json", "/kaggle/working/")

!ls -lh /kaggle/working/qwen2vl-2b-lora-v2.zip
print("\nDownload this from the Output panel, then unzip into checkpoints/ in the repo.")